# 01 — Explore Aya Teacher Model

**Goal**: Load Cohere Aya Expanse 8B, inspect architecture, profile resource usage, and analyze tokenizer behavior across all 10 target languages.

This notebook establishes the teacher baseline that our Aetheris student must match.

---

| Language | Family | Script | Code |
|----------|--------|--------|------|
| English | Indo-European | Latin | en |
| Spanish | Indo-European | Latin | es |
| Hindi | Indo-European | Devanagari | hi |
| Mandarin | Sino-Tibetan | CJK | zh |
| Arabic | Afroasiatic | Arabic | ar |
| Swahili | Niger-Congo | Latin | sw |
| Turkish | Turkic | Latin | tr |
| Japanese | Japonic | CJK+Kana | ja |
| Indonesian | Austronesian | Latin | id |
| Telugu | Dravidian | Telugu | te |

**Model**: `CohereForAI/aya-expanse-8b` (4-bit quantized for RTX 3050 4GB)

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import time
import json
import numpy as np
import psutil
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 1. Load Model (4-bit Quantized)

We use NF4 quantization via bitsandbytes to fit the 8B parameter model into ~4GB VRAM.

In [ ]:
MODEL_NAME = "CohereForAI/aya-expanse-8b"

# Memory before loading
mem_before = psutil.Process().memory_info().rss / 1024**3
gpu_before = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0

t0 = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

load_time = time.perf_counter() - t0
mem_after = psutil.Process().memory_info().rss / 1024**3
gpu_after = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0

print(f"Load time: {load_time:.1f}s")
print(f"RAM: {mem_before:.2f} -> {mem_after:.2f} GB (+{mem_after - mem_before:.2f})")
print(f"GPU: {gpu_before:.2f} -> {gpu_after:.2f} GB (+{gpu_after - gpu_before:.2f})")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

## 2. Inspect Architecture

Map out the Cohere model structure — layer types, projection dimensions, and where we'll hook in for distillation.

In [ ]:
# Model config
cfg = model.config
print("=== Model Config ===")
for key in ["hidden_size", "intermediate_size", "num_hidden_layers",
            "num_attention_heads", "num_key_value_heads", "max_position_embeddings",
            "vocab_size", "model_type"]:
    print(f"  {key}: {getattr(cfg, key, 'N/A')}")

print(f"\n=== Layer Structure (layer 0) ===")
layer0 = model.model.layers[0]
for name, module in layer0.named_modules():
    if name and "." not in name:
        params = sum(p.numel() for p in module.parameters())
        print(f"  {name}: {type(module).__name__} ({params:,} params)")

print(f"\n=== Attention Projections ===")
attn = layer0.self_attn
for proj_name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
    proj = getattr(attn, proj_name)
    print(f"  {proj_name}: {proj.weight.shape} ({proj.weight.dtype})")

print(f"\n=== MLP Projections ===")
mlp = layer0.mlp
for proj_name in ["gate_proj", "up_proj", "down_proj"]:
    proj = getattr(mlp, proj_name)
    print(f"  {proj_name}: {proj.weight.shape} ({proj.weight.dtype})")

print(f"\nGQA ratio: {cfg.num_attention_heads // cfg.num_key_value_heads}:1")
print(f"Head dim: {cfg.hidden_size // cfg.num_attention_heads}")

## 3. Tokenizer Analysis Across Languages

How efficiently does the Aya tokenizer encode each language? Higher tokens-per-word ratios indicate less efficient encoding — a source of inequity in multilingual models.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

SAMPLES = {
    "en": "The quick brown fox jumps over the lazy dog. Machine learning is a subset of artificial intelligence.",
    "es": "El rapido zorro marron salta sobre el perro perezoso. El aprendizaje automatico es un subconjunto de la inteligencia artificial.",
    "hi": "तेज भूरी लोमड़ी आलसी कुत्ते के ऊपर कूदती है। मशीन लर्निंग कृत्रिम बुद्धिमत्ता का एक उपसमुच्चय है।",
    "zh": "敏捷的棕色狐狸跳过了懒狗。机器学习是人工智能的一个子集。",
    "ar": "الثعلب البني السريع يقفز فوق الكلب الكسول. التعلم الآلي هو مجموعة فرعية من الذكاء الاصطناعي.",
    "sw": "Mbweha wa kahawia mwepesi anaruka juu ya mbwa mvivu. Ujifunzaji wa mashine ni sehemu ndogo ya akili bandia.",
    "tr": "Hizli kahverengi tilki tembel kopegin uzerinden atlar. Makine ogrenimi yapay zekanin bir alt kumesidir.",
    "ja": "素早い茶色の狐が怠けた犬を飛び越える。機械学習はデータから学習するシステムの構築に焦点を当てた人工知能のサブセットです。",
    "id": "Rubah cokelat cepat melompati anjing malas. Pembelajaran mesin adalah bagian dari kecerdasan buatan.",
    "te": "వేగవంతమైన గోధుమ రంగు నక్క సోమరి కుక్క మీదుగా దూకుతుంది. మెషిన్ లర్నింగ్ కృత్రిమ మేధస్సు యొక్క ఉపసమితి.",
}

LANG_NAMES = {
    "en": "English", "es": "Spanish", "hi": "Hindi", "zh": "Mandarin",
    "ar": "Arabic", "sw": "Swahili", "tr": "Turkish", "ja": "Japanese",
    "id": "Indonesian", "te": "Telugu",
}

rows = []
for lang, text in SAMPLES.items():
    tokens = tokenizer.encode(text)
    words = text.split()
    rows.append({
        "lang": lang,
        "language": LANG_NAMES[lang],
        "chars": len(text),
        "words": len(words),
        "tokens": len(tokens),
        "tok_per_word": len(tokens) / max(len(words), 1),
        "tok_per_char": len(tokens) / max(len(text), 1),
    })

df = pd.DataFrame(rows).set_index("lang")
print(df[["language", "words", "tokens", "tok_per_word", "tok_per_char"]].to_string())

# Plot tokenizer efficiency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.Set3(np.linspace(0, 1, len(df)))

ax = axes[0]
bars = ax.barh(df["language"], df["tok_per_word"], color=colors)
ax.set_xlabel("Tokens per Word")
ax.set_title("Tokenizer Efficiency by Language")
ax.axvline(x=df["tok_per_word"].median(), color="red", linestyle="--", alpha=0.7, label="median")
ax.legend()

ax = axes[1]
ax.barh(df["language"], df["tokens"], color=colors)
ax.set_xlabel("Total Tokens")
ax.set_title("Token Count for Same-Content Sentences")

plt.tight_layout()
plt.savefig("../results/tokenizer_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nMedian tok/word: {df['tok_per_word'].median():.2f}")
print(f"Max/min ratio: {df['tok_per_word'].max() / df['tok_per_word'].min():.2f}x")

## 4. Inference Profiling

Measure generation speed (tokens/sec) and time-to-first-token (TTFT) per language. This establishes the baseline our student must beat on throughput while staying close on quality.

In [ ]:
PROMPTS = {
    "en": "Explain the concept of gravity in simple terms.",
    "es": "Explica el concepto de gravedad en terminos sencillos.",
    "hi": "गुरुत्वाकर्षण की अवधारणा को सरल शब्दों में समझाइए।",
    "zh": "用简单的语言解释引力的概念。",
    "ar": "اشرح مفهوم الجاذبية بعبارات بسيطة.",
    "sw": "Eleza dhana ya mvutano kwa maneno rahisi.",
    "tr": "Yercekim kavramini basit terimlerle aciklayin.",
    "ja": "重力の概念を簡単な言葉で説明してください。",
    "id": "Jelaskan konsep gravitasi dengan istilah sederhana.",
    "te": "గురుత్వాకర్షణ భావనను సరళమైన పదాలలో వివరించండి.",
}

MAX_NEW_TOKENS = 64

# Warmup
inputs = tokenizer("Hello", return_tensors="pt").to(model.device)
_ = model.generate(**inputs, max_new_tokens=4, do_sample=False)

results = {}
for lang, prompt in PROMPTS.items():
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # TTFT
    t0 = time.perf_counter()
    _ = model.generate(**inputs, max_new_tokens=1, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    ttft = time.perf_counter() - t0

    # Full gen
    t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    gen_time = time.perf_counter() - t0

    output_len = out.shape[1] - input_len
    tok_per_sec = output_len / gen_time if gen_time > 0 else 0
    text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)

    results[lang] = {
        "language": LANG_NAMES[lang],
        "input_tokens": input_len,
        "output_tokens": output_len,
        "ttft_s": round(ttft, 3),
        "tok_per_sec": round(tok_per_sec, 1),
        "preview": text[:120],
    }
    print(f"{lang} ({LANG_NAMES[lang]:>10}): {tok_per_sec:5.1f} tok/s | TTFT={ttft:.3f}s | {input_len}→{output_len} tokens")

In [ ]:
# Visualize inference speed per language
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

langs = [r["language"] for r in results.values()]
tps = [r["tok_per_sec"] for r in results.values()]
ttfts = [r["ttft_s"] for r in results.values()]

ax = axes[0]
ax.barh(langs, tps, color=colors)
ax.set_xlabel("Tokens/sec")
ax.set_title("Generation Speed by Language (4-bit, RTX 3050)")

ax = axes[1]
ax.barh(langs, ttfts, color=colors)
ax.set_xlabel("Time to First Token (seconds)")
ax.set_title("TTFT by Language")

plt.tight_layout()
plt.savefig("../results/inference_profile.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Generation Samples

Quick sanity check — does the teacher produce reasonable outputs in each language?

In [ ]:
for lang, r in results.items():
    print(f"\n{'='*60}")
    print(f"  {r['language']} ({lang})")
    print(f"{'='*60}")
    print(f"  Prompt: {PROMPTS[lang]}")
    print(f"  Output: {r['preview']}...")

## 6. Save Results

In [ ]:
from pathlib import Path
import json

Path("../results").mkdir(exist_ok=True)

profile = {
    "model": MODEL_NAME,
    "load_time_s": round(load_time, 1),
    "ram_delta_gb": round(mem_after - mem_before, 2),
    "gpu_delta_gb": round(gpu_after - gpu_before, 2),
    "params": sum(p.numel() for p in model.parameters()),
    "tokenizer": df.to_dict(orient="index"),
    "inference": results,
}

with open("../results/01_teacher_profile.json", "w") as f:
    json.dump(profile, f, indent=2, default=str)

print("Saved to results/01_teacher_profile.json")